#**Multi-Agent Intelligent Academic Course Advisor System**

## 📊 Dataset Description: **Udemy Courses Dataset (2022)**

The system uses the **[Udemy Courses Dataset](https://www.kaggle.com/datasets/hossaingh/udemy-courses)** from Kaggle, which contains course-level data scraped from Udemy. The main CSV used is `Course_info.csv`.

### ✅ Key Features (Columns Used):

| Column | Description |
|--------|-------------|
| `title` | Course title |
| `headline` | Short subtitle/summary |
| `price` | Course price (USD) |
| `avg_rating` | Average rating (0.0–5.0) |
| `num_reviews` | Total number of reviews |
| `num_subscribers` | Number of enrolled students |
| `content_length_min` | Total duration in minutes |
| `category` | Broad category (e.g., Business, IT) |
| `topic` | Specific topic keywords (optional/partial) |
| `language` | Course language |
| `instructor_name` | Instructor’s name |
| `course_url` | Link to course page |

---

## 🤖 The 5 Intelligent Agents

### 1. **Input Agent**
- **Goal**: Understands vague, natural language queries from the user.
- **How**: Uses a Hugging Face-hosted LLM (`flan-t5-base`) to extract structured filters like:
  - `topic`, `category`, `max_price`, `language`, `duration`
- **Output**: A JSON-like dictionary of preferences.

---

### 2. **Retriever Agent**
- **Goal**: Retrieves relevant courses based on the user’s input.
- **Two versions**:
  - **Basic version**: Uses Pandas filters and ranking (by rating/reviews/subscribers).
  - **Advanced version**: Uses **FAISS + Sentence Transformers** for semantic vector search (RAG-style).

---

### 3. **Q&A Agent**
- **Goal**: Answers follow-up questions about recommended courses.
- **How**: Uses an LLM to generate helpful responses based on the metadata of the course (e.g., duration, language, rating).

---

### 4. **Recommender Agent**
- **Goal**: Selects and ranks the top courses to recommend.
- **Basic version**: Logic-based ranking using `avg_rating`, `num_reviews`, etc.
- **Advanced version**: LLM-generated persuasive justifications for each course, tailored to the user’s preferences.

---

### 5. **Feedback Agent**
- **Goal**: Collects feedback (rating and satisfaction) from the user.
- **Basic version**: Captures thumbs up/down and a 1–5 rating.
- **Advanced option**: Uses LLM to classify free-text feedback sentiment.

---

## 🖥️ Gradio User Interface (UI)

To create an intuitive and interactive experience, a **Gradio interface** was developed. It allows the user to:
- Enter a free-text query (e.g., *"I want a short AI course in Arabic under $30"*)
- Ask optional follow-up questions (e.g., *"Is this course beginner-friendly?"*)
- View top 3 recommended courses with detailed explanations
- Optionally, give feedback for continuous improvement

This makes the system feel like a smart chatbot-powered academic advisor.

---



## ✅ Final Outcome

The final system simulates a **realistic AI-powered academic advisor**, capable of:
- Parsing vague user needs
- Retrieving and ranking courses intelligently
- Justifying recommendations conversationally
- Accepting and learning from user feedback

The modular design enables easy extension with more advanced models or additional logic layers like personalization or multilingual support.

---



##**Data Preprocessing**

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Course_info.csv to Course_info.csv


In [2]:
import pandas as pd
# Load the uploaded file
df = pd.read_csv("Course_info.csv")
df.head(10)

,id,title,is_paid,price,headline,num_subscribers,avg_rating,num_reviews,num_comments,num_lectures,content_length_min,published_time,last_update_date,category,subcategory,topic,language,course_url,instructor_name,instructor_url
0,4715.0,Online Vegan Vegetarian Cooking School,True,24.99,Learn to cook delicious vegan recipes. Filmed ...,2231.0,3.75000,134.0,42.0,37.0,1268.0,2010-08-05T22:06:13Z,2020-11-06,Lifestyle,Food & Beverage,Vegan Cooking,English,/course/vegan-vegetarian-cooking-school/,Angela Poch,/user/angelapoch/
1,1769.0,The Lean Startup Talk at Stanford E-Corner,False,0.00,Debunking Myths of Entrepreneurship A startup ...,26474.0,4.50000,709.0,112.0,9.0,88.0,2010-01-12T18:09:46Z,NaN,Business,Entrepreneurship,Lean Startup,English,/course/the-lean-startup-debunking-myths-of-en...,Eric Ries,/user/ericries/
2,5664.0,"How To Become a Vegan, Vegetarian, or Flexitarian",True,19.99,Get the tools you need for a lifestyle change ...,1713.0,4.40000,41.0,13.0,14.0,82.0,2010-10-13T18:07:17Z,2019-10-09,Lifestyle,Other Lifestyle,Vegan Cooking,English,/course/see-my-personal-motivation-for-becomin...,Angela Poch,/user/angelapoch/
3,7723.0,How to Train a Puppy,True,199.99,Train your puppy the right way with Dr. Ian Du...,4988.0,4.80000,395.0,88.0,36.0,1511.0,2011-06-20T20:08:38Z,2016-01-13,Lifestyle,Pet Care & Training,Pet Training,English,/course/complete-dunbar-collection/,Ian Dunbar,/user/ian-dunbar/
4,8157.0,Web Design from the Ground Up,True,159.99,Learn web design online: Everything you need t...,1266.0,4.75000,38.0,12.0,38.0,569.0,2011-06-23T18:31:20Z,NaN,Design,Web Design,Web Design,English,/course/web-design-from-the-ground-up/,E Learning Lab,/user/edwin-ang-2/
5,8139.0,14-Day Yoga Detox and Empowerment Course,True,29.99,"Lose weight, get healthier and fit on all leve...",20505.0,4.53012,796.0,135.0,31.0,1163.0,2011-07-15T04:13:24Z,2018-05-22,Health & Fitness,Yoga,Yoga,English,/course/yoga-for-weight-loss-and-core-strength...,Sadie Nardini,/user/sadienardini/
6,2762.0,Simple Strategy for Swing Trading the Stock Ma...,True,39.99,Use my favorite Technical Indicator and the Tr...,3309.0,3.85000,958.0,241.0,8.0,80.0,2010-04-14T16:32:46Z,2019-03-07,Finance & Accounting,Investing & Trading,Swing Trading,English,/course/swing-trading-the-stock-market/,Tom Watson,/user/tomwatson/
7,8082.0,Ruby Programming for Beginners,True,74.99,Learn Ruby Programming the fast and easy way!,28824.0,4.00000,741.0,189.0,56.0,363.0,2011-07-08T21:32:55Z,2022-09-26,Development,Programming Languages,Ruby,English,/course/learn-ruby-programming-in-ten-easy-steps/,Huw Collingbourne,/user/huwcollingbourne/
8,8075.0,How to Create an Awesome Demo Video for Your B...,True,149.99,You don't need to spend $10K in order to have ...,10761.0,3.90000,349.0,101.0,87.0,526.0,2011-07-06T14:06:34Z,2020-11-22,Business,Media,Demo Video,English,/course/how-to-create-awesome-demo-videos/,Miguel Hernandez,/user/miguelhernandez/
9,8069.0,Curso SEO Online,True,99.99,Curso SEO práctico. Aprenda a posicionar su si...,483.0,4.65000,100.0,45.0,73.0,373.0,2012-07-03T17:03:28Z,2020-02-28,Marketing,Search Engine Optimization,SEO,Spanish,/course/curso-de-posicionamiento-en-buscadores...,Juan Jose Ramos,/user/juanjo-ramos/


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209734 entries, 0 to 209733
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  209734 non-null  float64
 1   title               209734 non-null  object 
 2   is_paid             209734 non-null  bool   
 3   price               209734 non-null  float64
 4   headline            209703 non-null  object 
 5   num_subscribers     209734 non-null  float64
 6   avg_rating          209734 non-null  float64
 7   num_reviews         209734 non-null  float64
 8   num_comments        209734 non-null  float64
 9   num_lectures        209734 non-null  float64
 10  content_length_min  209734 non-null  float64
 11  published_time      209734 non-null  object 
 12  last_update_date    209597 non-null  object 
 13  category            209734 non-null  object 
 14  subcategory         209734 non-null  object 
 15  topic               208776 non-nul

In [4]:
print(f"Dataset shape: {df.shape}")

Dataset shape: (209734, 20)


In [5]:
print(df.isnull().sum())

id                      0
title                   0
is_paid                 0
price                   0
headline               31
num_subscribers         0
avg_rating              0
num_reviews             0
num_comments            0
num_lectures            0
content_length_min      0
published_time          0
last_update_date      137
category                0
subcategory             0
topic                 958
language                0
course_url              0
instructor_name         5
instructor_url        427
dtype: int64


In [6]:
# Fill textual columns with default values
df['headline'] = df['headline'].fillna('')
df['last_update_date'] = df['last_update_date'].fillna('Unknown')
df['topic'] = df['topic'].fillna('Unknown')
df['instructor_name'] = df['instructor_name'].fillna('Unknown Instructor')
df['instructor_url'] = df['instructor_url'].fillna('')

In [7]:
#Confirm there are no more nulls
print(df.isnull().sum())

id                    0
title                 0
is_paid               0
price                 0
headline              0
num_subscribers       0
avg_rating            0
num_reviews           0
num_comments          0
num_lectures          0
content_length_min    0
published_time        0
last_update_date      0
category              0
subcategory           0
topic                 0
language              0
course_url            0
instructor_name       0
instructor_url        0
dtype: int64


In [8]:
print(df.columns)

Index(['id', 'title', 'is_paid', 'price', 'headline', 'num_subscribers',
       'avg_rating', 'num_reviews', 'num_comments', 'num_lectures',
       'content_length_min', 'published_time', 'last_update_date', 'category',
       'subcategory', 'topic', 'language', 'course_url', 'instructor_name',
       'instructor_url'],
      dtype='object')


In [9]:
# Drop duplicates (if any)
df = df.drop_duplicates()

In [10]:
# Remove rows with missing essential info
df = df.dropna(subset=['title', 'category', 'price', 'content_length_min'])

In [11]:
# Clean up column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [12]:
# Convert price and content length to numeric (if not already)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['content_length_min'] = pd.to_numeric(df['content_length_min'], errors='coerce')

In [13]:
# Create synthetic 'description' for embedding purposes
df['description'] = df['title'] + " - " + df['headline']

In [14]:
df['description'] = df['description'].fillna('')

In [15]:
# Check unique values in real columns
print(df['category'].unique())      # Instead of 'subject'
print(df['subcategory'].unique())   # Optional, for more detail
print(df['topic'].unique())         # Optional, some may be null

['Lifestyle' 'Business' 'Design' 'Health & Fitness' 'Finance & Accounting'
 'Development' 'Marketing' 'Teaching & Academics' 'IT & Software'
 'Office Productivity' 'Music' 'Personal Development'
 'Photography & Video']
['Food & Beverage' 'Entrepreneurship' 'Other Lifestyle'
 'Pet Care & Training' 'Web Design' 'Yoga' 'Investing & Trading'
 'Programming Languages' 'Media' 'Search Engine Optimization'
 'Teacher Training' 'Mobile Development' 'IT Certifications'
 'Web Development' 'Software Development Tools' 'Design Tools' 'Science'
 'General Health' 'Communication' 'Microsoft' 'Digital Marketing'
 'Language Learning' 'Dance' 'Hardware' 'Marketing Fundamentals'
 'Database Design & Development' 'Sales'
 'Business Analytics & Intelligence' 'Game Development' 'Google'
 'Industry' 'Humanities' 'Oracle' 'Content Marketing'
 'Social Media Marketing' 'Other Teaching & Academics' 'Social Science'
 'Operations' 'Instruments' 'Business Strategy' 'Career Development'
 'Arts & Crafts' 'Parenting & Re

##**Agent 1**

### 🤖 simple Input Agent – Goal

The **Input Agent** is responsible for:
- Taking in a **natural language query** from the user
- **Extracting relevant filters**, such as:
  - Topic or keywords
  - Price range
  - Language
  - Category (e.g., Business, IT)
  - Course length
  - Level (if inferred from text)

---

## 🧠 Steps

1. Accept user input as text
2. Extract filters using basic NLP (e.g., keyword matching or regex)
3. Return a dictionary of the extracted preferences

---

###1. Define the keywords

In [16]:
# Define lists of known values from your dataset for matching
categories = df['category'].unique().tolist()
languages = df['language'].unique().tolist()
topics = df['topic'].unique().tolist()

###2.Sample Input Agent Function

In [17]:
import re

def input_agent(user_input):
    filters = {
        'category': None,
        'topic': None,
        'max_price': None,
        'language': None,
        'min_duration': None,
        'max_duration': None
    }

    # Lowercase input
    text = user_input.lower()

    # Category matching
    for cat in categories:
        if cat.lower() in text:
            filters['category'] = cat
            break

    # Topic matching
    for topic in topics:
        if topic.lower() in text:
            filters['topic'] = topic
            break

    # Language matching
    for lang in languages:
        if lang.lower() in text:
            filters['language'] = lang
            break

    # Price extraction
    price_match = re.search(r'under \$?(\d+)', text)
    if price_match:
        filters['max_price'] = float(price_match.group(1))

    # Duration (hours or minutes)
    duration_match = re.search(r'under (\d+)\s*(min|minutes|hours|hrs)', text)
    if duration_match:
        value = int(duration_match.group(1))
        unit = duration_match.group(2)
        filters['max_duration'] = value if 'min' in unit else value * 60

    return filters

###3.Try Example Inputs

In [18]:
query1 = "I want a business course under $30 in English"
query2 = "Show me data science courses in arabic under 2 hours"

print(input_agent(query1))
print(input_agent(query2))

{'category': 'Business', 'topic': None, 'max_price': 30.0, 'language': 'English', 'min_duration': None, 'max_duration': None}
{'category': None, 'topic': 'Data Science', 'max_price': 2.0, 'language': 'Arabic', 'min_duration': None, 'max_duration': 120}


## **LLM-Powered Agent**

###Setup LLM

In [19]:
!pip install -q transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.9 MB/s eta 0:00:00


In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_id = "declare-lab/flan-alpaca-large"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

llm = pipeline("text2text-generation", model=model, tokenizer=tokenizer)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Device set to use cuda:0


In [22]:
#Modify the Prompt (cleaner + enforce structure)
def build_prompt(user_query):
    return f"""
You are an academic advisor AI assistant.

Extract these fields from the following request and return a JSON object using double quotes and valid syntax:

- category
- topic
- max_price (USD)
- language
- duration (in minutes)

If any field is missing or unclear, return null for it.

Request: "{user_query}"

Respond ONLY with JSON, like:
{{
  "category": "IT",
  "topic": "Artificial Intelligence",
  "max_price": 50,
  "language": "Arabic",
  "duration": 180
}}
"""


In [23]:
#Update Model Initialization (if using flan-alpaca-large)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
model_id = "declare-lab/flan-alpaca-large"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
llm = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [24]:
import json
import re

def parse_response(response_text):
    try:
        # Try extracting a JSON block
        json_str = re.search(r"\{.*\}", response_text, re.DOTALL)
        if json_str:
            return json.loads(json_str.group())

        # If no { } found, but the text contains fields, wrap it manually
        if ":" in response_text and "{" not in response_text:
            fixed = "{" + response_text.strip().rstrip(",") + "}"
            return json.loads(fixed)

        # Nothing valid found
        print("⚠️ LLM returned unstructured output:")
        print(response_text)
        return None

    except json.JSONDecodeError:
        print("⚠️ Still couldn't parse. Output was:")
        print(response_text)
        return None

In [26]:
def llm_input_agent(user_query):
    """
    Uses the LLM to extract filters from a user query.
    """
    prompt = build_prompt(user_query)
    response = llm(prompt, max_length=200)[0]['generated_text']
    return response

query = "I need a short Arabic AI course under 50 dollars"
response = llm_input_agent(query)
print(response)

parsed_filters = parse_response(response)
print(parsed_filters)

 "category": "IT", "topic": "Artificial Intelligence", "max_price": 50, "language": "Arabic", "duration": 180 
{'category': 'IT', 'topic': 'Artificial Intelligence', 'max_price': 50, 'language': 'Arabic', 'duration': 180}


##**Agent 2: Retriever Agent 🔍**

In [28]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 52.7 MB/s eta 0:00:00


In [29]:
#Create Embeddings from the Dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")  # Fast + accurate

# Prepare course texts (you can change description field if needed)
df['text'] = df['title'] + ' - ' + df['headline']

# Generate embeddings
course_embeddings = embedding_model.encode(df['text'].tolist(), show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6555 [00:00<?, ?it/s]

In [30]:
#Build FAISS Index
dimension = course_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(course_embeddings))  # Add all course vectors to the index

In [31]:
#Define the LLM-Powered Retriever Agent
def rag_retriever_agent(user_query, top_k=5):
    # Convert query to vector
    query_embedding = embedding_model.encode([user_query])

    # Search in FAISS index
    distances, indices = index.search(np.array(query_embedding), top_k)

    # Get matching courses
    matches = df.iloc[indices[0]].copy()
    matches['similarity'] = 1 - distances[0] / distances[0].max()  # Optional similarity score
    return matches.reset_index(drop=True)


In [32]:
query = "I want something like a crash course on AI"
results = rag_retriever_agent(query, top_k=5)

print(results[['title', 'headline', 'similarity']])


                                               title  \
0               Welcome to Artificial Intelligence !   
1  Great Start for Coding - Python Crash Course f...   
2  Artificial Intelligence Expert Course + Live C...   
3  Learn Machine Learning 101 Class Bootcamp Cour...   
4                      Machine Learning Crash Course   

                                            headline  similarity  
0  NON TECHNICAL COURSE specifically created for ...    0.081695  
1        Python Crash Course for Data Science and AI    0.057662  
2  A breathtaking course in 2022 that teaches new...    0.027818  
3  Machine Learning 101 Class Bootcamp Course Int...    0.013969  
4  Learn the Core of Machine Learning in just 2 h...    0.000000  


##**Agent 3: Q&A Agent 🔍💬**

In [46]:
#Define a Prompt Function
def build_qa_prompt(course_row, user_question):
    return f"""
You are a helpful course assistant.

Answer the student's question about the following course based only on the course info.

Course Info:
- Title: {course_row['title']}
- Headline: {course_row['headline']}
- Price: ${course_row['price']}
- Average Rating: {course_row['avg_rating']}
- Number of Subscribers: {course_row['num_subscribers']}
- Duration: {course_row['content_length_min']} minutes
- Language: {course_row['language']}

Student question: "{user_question}"

Respond in a helpful and polite tone.
"""


In [47]:
#Q&A Agent Function
def qa_agent(course_row, user_question, llm):
    prompt = build_qa_prompt(course_row, user_question)
    response = llm(prompt, max_new_tokens=150, do_sample=False)[0]['generated_text']
    return response.strip()

In [48]:
#Example
# Assume this is your top match from the retriever
# Call the retriever function and store results
retrieved_courses = rag_retriever_agent(query, top_k=5)
course = retrieved_courses.iloc[0]

# User's follow-up question
user_q = "Is this course suitable for beginners?"

# Get LLM response
answer = qa_agent(course, user_q, llm)
print("💬 Q&A Agent Response:\n", answer)

💬 Q&A Agent Response:
 Yes, this course is suitable for beginners. It is a non-technical course that provides an overview of the basics of AI/ML/DL and provides an overview of the road map to A.I.


In [49]:
#Multicourse Q&A agent
def multi_course_qa_agent(retrieved_df, user_question, llm, top_n=3):
    responses = []

    for i in range(min(top_n, len(retrieved_df))):
        course = retrieved_df.iloc[i]
        prompt = f"""
You are an academic course assistant.

Here is the information about a course:
- Title: {course['title']}
- Headline: {course['headline']}
- Price: ${course['price']}
- Rating: {course['avg_rating']}
- Subscribers: {course['num_subscribers']}
- Duration: {course['content_length_min']} minutes
- Language: {course['language']}

Student's question: "{user_question}"

Respond with a helpful answer about **this course only**. Keep it brief and informative.
"""
        answer = llm(prompt, max_new_tokens=150, do_sample=False)[0]['generated_text']
        responses.append({
            "title": course['title'],
            "answer": answer.strip()
        })

    return responses

In [50]:
user_q = "Is this course beginner-friendly and worth buying?"
answers = multi_course_qa_agent(retrieved_courses, user_q, llm, top_n=3)

for i, response in enumerate(answers):
    print(f"\n📘 Course {i+1}: {response['title']}\n💬 Answer: {response['answer']}")


📘 Course 1: Welcome to Artificial Intelligence !
💬 Answer: This course is beginner-friendly and worth buying. It provides an overview of the basics of Artificial Intelligence, including its history, development, and applications. It is designed for AI/ML/DL aspirants and provides an overview of the road map to A.I. It is also designed to provide an overview of the current state of AI and its potential applications.

📘 Course 2: Great Start for Coding - Python Crash Course for Beginners
💬 Answer: This course is beginner-friendly and worth buying. It covers the basics of Python, including syntax, data structures, and algorithms. It is designed for those who are just starting out in programming and is rated 3.75 by the subscribers. It is also available in English and has a duration of 95.0 minutes.

📘 Course 3: Artificial Intelligence Expert Course + Live Class
💬 Answer: This course is beginner-friendly and worth buying. It teaches new-age Artificial Intelligence (AI) Technologies and To

##**Agent 4: Recommender Agent 🎯**

In [53]:
#Define the Prompt Function
def build_recommendation_prompt(course_row, user_preferences):
    return f"""
You are a helpful academic advisor assistant.

Based on the student's preferences:
- Topic: {user_preferences.get('topic')}
- Category: {user_preferences.get('category')}
- Max Price: ${user_preferences.get('max_price')}
- Language: {user_preferences.get('language')}
- Duration limit: {user_preferences.get('duration')} minutes

Recommend the following course persuasively:

Course:
- Title: {course_row['title']}
- Headline: {course_row['headline']}
- Price: ${course_row['price']}
- Rating: {course_row['avg_rating']}
- Reviews: {course_row['num_reviews']}
- Subscribers: {course_row['num_subscribers']}
- Duration: {course_row['content_length_min']} minutes
- Language: {course_row['language']}

Write a short recommendation in a friendly and confident tone.
"""


In [54]:
#LLM-Powered Recommender Function
def llm_recommender_agent(retrieved_df, user_preferences, llm, top_n=3):
    recommendations = []
    top_courses = retrieved_df.head(top_n)

    for _, course in top_courses.iterrows():
        prompt = build_recommendation_prompt(course, user_preferences)
        response = llm(prompt, max_new_tokens=150, do_sample=False)[0]['generated_text']

        recommendations.append({
            "title": course['title'],
            "generated_reason": response.strip(),
            "price": course['price'],
            "duration": course['content_length_min'],
            "language": course['language']
        })

    return recommendations

In [55]:
user_preferences = {
    'topic': 'Artificial Intelligence',
    'category': 'IT',
    'max_price': 50,
    'language': 'Arabic',
    'duration': 180
}

recommendations = llm_recommender_agent(retrieved_courses, user_preferences, llm, top_n=3)

for i, rec in enumerate(recommendations):
    print(f"\n⭐ Recommendation {i+1}")
    print(f"🎓 Title: {rec['title']}")
    print(f"💬 Reason: {rec['generated_reason']}")
    print(f"💵 Price: ${rec['price']}")
    print(f"🕒 Duration: {rec['duration']} minutes")
    print(f"🌐 Language: {rec['language']}")


⭐ Recommendation 1
🎓 Title: Welcome to Artificial Intelligence !
💬 Reason: I would highly recommend the course "Ways to Make the Most of Artificial Intelligence" by Professor John Smith. This course is designed to provide an in-depth look at the fundamentals of AI, ML, and DLL development, and provides an in-depth look at the road map to A.I. It is a great way to get an understanding of the fundamentals of AI and gain a better understanding of the field.
💵 Price: $0.0
🕒 Duration: 49.0 minutes
🌐 Language: English

⭐ Recommendation 2
🎓 Title: Great Start for Coding - Python Crash Course for Beginners
💬 Reason: I would highly recommend the Great Start for Coding - Python Crash Course for Beginners. This course is a great way to learn the basics of Python and get started on your journey towards becoming an AI expert. It is a great way to get started with the basics of data science and AI, and it is a great way to learn the basics of programming in Arabic. It is also a great way to learn t

##**Agent 5: Feedback Agent 🔁🎯**

In [56]:
#Feedback Agent Function
def feedback_agent(course_title, question_type="recommendation"):
    print(f"\n📝 Feedback Requested for: {course_title}")

    if question_type == "recommendation":
        feedback = input("Was this course recommendation helpful? (yes/no): ").strip().lower()
        rating = input("How would you rate it from 1 to 5? ")
    elif question_type == "qa":
        feedback = input("Was the answer to your question helpful? (yes/no): ").strip().lower()
        rating = input("How would you rate the answer from 1 to 5? ")
    else:
        feedback = "unknown"
        rating = "0"

    return {
        "course_title": course_title,
        "feedback": feedback,
        "rating": int(rating) if rating.isdigit() else 0,
        "type": question_type
    }

In [57]:
# After showing a recommended course
feedback_data = feedback_agent(course_title="AI Crash Course", question_type="recommendation")
print("\n✅ Feedback Recorded:")
print(feedback_data)


📝 Feedback Requested for: AI Crash Course
Was this course recommendation helpful? (yes/no): yes
How would you rate it from 1 to 5? 4

✅ Feedback Recorded:
{'course_title': 'AI Crash Course', 'feedback': 'yes', 'rating': 4, 'type': 'recommendation'}


In [58]:
#Store Feedback in a Log
import pandas as pd
# Feedback history list
feedback_log = []
# Append new feedback
feedback_log.append(feedback_data)
# Convert to DataFrame
feedback_df = pd.DataFrame(feedback_log)

## **Gradio UI**

In [61]:
!pip install gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.4 MB/s eta 0:00:00


In [62]:
def advisor_system(user_query, follow_up_question):
    # 1. Input Agent
    filters = llm_input_agent(user_query)
    parsed = parse_response(filters)
    if not parsed:
        return "❌ Sorry, could not understand your query."

    # 2. Retriever Agent (RAG or keyword-based)
    retrieved_df = rag_retriever_agent(user_query)  # or use retriever_agent(parsed, df)

    if retrieved_df.empty:
        return "❌ No matching courses found."

    # 3. Recommender Agent (LLM-powered)
    recommendations = llm_recommender_agent(retrieved_df, parsed, llm, top_n=3)

    # 4. Q&A Agent (if follow-up provided)
    qa_response = ""
    if follow_up_question:
        course = retrieved_df.iloc[0]
        qa_response = qa_agent(course, follow_up_question, llm)

    # Format response
    result = ""
    for i, rec in enumerate(recommendations):
        result += f"\n⭐ Recommendation {i+1}\n"
        result += f"🎓 {rec['title']}\n💬 {rec['generated_reason']}\n"

    if follow_up_question:
        result += f"\n\n💡 Q&A Response:\n{qa_response}"

    return result

In [63]:
def advisor_system(user_query, follow_up_question):
    # 1. Input Agent
    filters = llm_input_agent(user_query)
    parsed = parse_response(filters)
    if not parsed:
        return "❌ Sorry, could not understand your query."

    # 2. Retriever Agent (RAG or keyword-based)
    retrieved_df = rag_retriever_agent(user_query)  # or use retriever_agent(parsed, df)

    if retrieved_df.empty:
        return "❌ No matching courses found."

    # 3. Recommender Agent (LLM-powered)
    recommendations = llm_recommender_agent(retrieved_df, parsed, llm, top_n=3)

    # 4. Q&A Agent (if follow-up provided)
    qa_response = ""
    if follow_up_question:
        course = retrieved_df.iloc[0]
        qa_response = qa_agent(course, follow_up_question, llm)

    # Format response
    result = ""
    for i, rec in enumerate(recommendations):
        result += f"\n⭐ Recommendation {i+1}\n"
        result += f"🎓 {rec['title']}\n💬 {rec['generated_reason']}\n"

    if follow_up_question:
        result += f"\n\n💡 Q&A Response:\n{qa_response}"

    return result


In [64]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 AI Course Advisor")

    user_input = gr.Textbox(label="📝 What kind of course are you looking for?")
    follow_up = gr.Textbox(label="💬 Optional: Ask a follow-up question")

    output = gr.Textbox(label="📋 System Response", lines=10)

    btn = gr.Button("🔍 Get Recommendations")
    btn.click(fn=advisor_system, inputs=[user_input, follow_up], outputs=output)

demo.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4602e6118ca0b57717.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
